# 04 — Validate Modeling Table & Engineer Pre-Departure Features

This notebook validates the merged OGG flight + weather + storm dataset and creates a first training-ready feature table for disruption prediction.

The main principle is **no target leakage**: features used for prediction should be available at or before the scheduled OGG event time.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if (cwd / 'data').exists():
    ROOT = cwd
elif (cwd.parent / 'data').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(f'Cannot locate project root from {cwd}')

MODEL_FILE = ROOT / 'data/processed/ogg_flight_weather_storm_2020_2026.csv.gz'
OUT_FILE = ROOT / 'data/processed/ogg_model_features_v1.csv.gz'
SCHEMA_FILE = ROOT / 'data/processed/ogg_model_features_v1_schema.json'

print('ROOT:', ROOT)
print('Modeling table exists:', MODEL_FILE.exists())


## 1. Load and inspect the merged modeling table


In [ ]:
df = pd.read_csv(MODEL_FILE, low_memory=False)
print('Shape:', df.shape)
print('Columns:', len(df.columns))
display(pd.DataFrame({'column': df.columns}))
display(df.head())


## 2. Validate target distribution, date range, and duplicate rows


In [ ]:
if 'FlightDate' in df.columns:
    df['FlightDate'] = pd.to_datetime(df['FlightDate'], errors='coerce')
    print('Flight range:', df['FlightDate'].min(), 'to', df['FlightDate'].max())

if 'disruption_class' not in df.columns:
    raise KeyError('Expected target column disruption_class was not found.')

target_counts = df['disruption_class'].value_counts(dropna=False)
target_pct = (df['disruption_class'].value_counts(dropna=False, normalize=True) * 100).round(3)
display(pd.concat([target_counts.rename('count'), target_pct.rename('percent')], axis=1))

dup_subset = [c for c in ['FlightDate','Reporting_Airline','Flight_Number_Reporting_Airline','Origin','Dest','ogg_sched_dt'] if c in df.columns]
if dup_subset:
    duplicate_count = int(df.duplicated(subset=dup_subset).sum())
    print('Potential duplicate flight rows:', duplicate_count)


## 3. Missingness and data coverage


In [ ]:
missing = df.isna().mean().mul(100).sort_values(ascending=False).rename('missing_pct').to_frame()
display(missing.head(40).round(2))

if 'weather_dt' in df.columns:
    weather_match_rate = 100 * df['weather_dt'].notna().mean()
    print('Overall weather match rate (%):', round(weather_match_rate, 2))

if 'FlightDate' in df.columns:
    df['year'] = df['FlightDate'].dt.year
    coverage_aggs = {'rows': ('FlightDate','size')}
    if 'weather_dt' in df.columns:
        coverage_aggs['weather_match_pct'] = ('weather_dt', lambda s: 100 * s.notna().mean())
    if 'storm_active_maui' in df.columns:
        coverage_aggs['storm_active_pct'] = ('storm_active_maui', lambda s: 100 * pd.Series(s).fillna(False).astype(bool).mean())
    yearly_coverage = df.groupby('year').agg(**coverage_aggs)
    display(yearly_coverage.round(2))


## 4. Identify leakage-prone columns

These variables are outcomes or are only known after the flight has already departed/arrived. They should **not** be used in the initial pre-departure predictor.


In [ ]:
leakage_candidates = [
    'DepTime', 'ArrTime', 'DepDelay', 'DepDelayMinutes', 'ArrDelay', 'ArrDelayMinutes',
    'Cancelled', 'CancellationCode', 'Diverted', 'ActualElapsedTime', 'AirTime',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
    'disruption_class'
]
leakage_cols = [c for c in leakage_candidates if c in df.columns]
display(pd.DataFrame({'leakage_or_target_column': leakage_cols}))


## 5. Build safe calendar, route, airline, and weather features

The first baseline uses only variables available before the scheduled OGG event. Historical airline/airport behavior can be added later using strictly past-only rolling features.


In [ ]:
work = df.copy()

# Normalize scheduled timestamp if present.
if 'ogg_sched_dt' in work.columns:
    work['ogg_sched_dt'] = pd.to_datetime(work['ogg_sched_dt'], errors='coerce')
    work['sched_hour'] = work['ogg_sched_dt'].dt.hour
    work['sched_dow'] = work['ogg_sched_dt'].dt.dayofweek
    work['sched_month'] = work['ogg_sched_dt'].dt.month
    work['sched_dayofyear'] = work['ogg_sched_dt'].dt.dayofyear
    work['is_weekend'] = work['sched_dow'].isin([5, 6]).astype(int)

# Cyclical encodings for hour and month.
if 'sched_hour' in work.columns:
    work['hour_sin'] = np.sin(2 * np.pi * work['sched_hour'] / 24)
    work['hour_cos'] = np.cos(2 * np.pi * work['sched_hour'] / 24)
if 'sched_month' in work.columns:
    work['month_sin'] = np.sin(2 * np.pi * work['sched_month'] / 12)
    work['month_cos'] = np.cos(2 * np.pi * work['sched_month'] / 12)

# Direction and route counterpart airport.
if 'direction' not in work.columns and {'Origin','Dest'}.issubset(work.columns):
    work['direction'] = np.where(work['Origin'].eq('OGG'), 'departure', 'arrival')
if {'Origin','Dest'}.issubset(work.columns):
    work['other_airport'] = np.where(work['Origin'].eq('OGG'), work['Dest'], work['Origin'])

# COVID-period flag retained only for analysis/sensitivity checks.
if 'FlightDate' in work.columns:
    work['covid_era'] = work['FlightDate'].between('2020-03-01', '2021-05-31').astype(int)

# Standardize boolean storm flag.
if 'storm_active_maui' in work.columns:
    work['storm_active_maui'] = work['storm_active_maui'].fillna(False).astype(int)


## 6. Candidate feature inventory


In [ ]:
categorical_candidates = [
    'Reporting_Airline', 'direction', 'other_airport',
]
numeric_candidates = [
    'Distance', 'CRSDepTime', 'CRSArrTime',
    'sched_hour', 'sched_dow', 'sched_month', 'sched_dayofyear', 'is_weekend',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'covid_era',
    'weather_age_min', 'storm_active_maui',
]

# Add cleaned NOAA numeric predictors created in Notebook 03.
numeric_candidates += [c for c in work.columns if c.endswith('_num')]

categorical_features = [c for c in categorical_candidates if c in work.columns]
numeric_features = list(dict.fromkeys([c for c in numeric_candidates if c in work.columns]))

feature_inventory = pd.DataFrame({
    'feature': categorical_features + numeric_features,
    'type': ['categorical'] * len(categorical_features) + ['numeric'] * len(numeric_features),
})
display(feature_inventory)
print('Categorical features:', len(categorical_features))
print('Numeric features:', len(numeric_features))


## 7. Target and feature completeness


In [ ]:
target_order = ['normal', 'delay', 'severe_delay', 'cancelled']
model_df = work[work['disruption_class'].isin(target_order)].copy()

feature_cols = categorical_features + numeric_features
feature_missing = model_df[feature_cols].isna().mean().mul(100).sort_values(ascending=False).rename('missing_pct').to_frame()
display(feature_missing.round(2))

print('Rows before target filtering:', len(work))
print('Rows after target filtering:', len(model_df))
display(model_df['disruption_class'].value_counts().reindex(target_order).fillna(0).astype(int).to_frame('count'))


## 8. Recommended temporal split

For this project we should avoid a random split. A time-based split better represents deployment and reduces temporal leakage.

Initial proposal:
- **Train:** 2020–2023
- **Validation:** 2024
- **Test:** 2025

2026 is currently excluded from the main baseline because the historical weather download does not yet cover the full year.


In [ ]:
if 'FlightDate' not in model_df.columns:
    raise KeyError('FlightDate is required for temporal splitting.')

model_df['split'] = np.select(
    [
        model_df['FlightDate'].dt.year <= 2023,
        model_df['FlightDate'].dt.year == 2024,
        model_df['FlightDate'].dt.year == 2025,
    ],
    ['train', 'validation', 'test'],
    default='holdout_or_uncovered',
)
display(pd.crosstab(model_df['split'], model_df['disruption_class'], margins=True))


## 9. Quick weather-disruption sanity checks


In [ ]:
weather_summary_cols = [c for c in [
    'HourlyWindSpeed_num', 'HourlyWindGustSpeed_num', 'HourlyVisibility_num',
    'HourlyPrecipitation_num', 'storm_active_maui'
] if c in model_df.columns]

if weather_summary_cols:
    display(model_df.groupby('disruption_class')[weather_summary_cols].mean(numeric_only=True).round(3))

if 'storm_active_maui' in model_df.columns:
    storm_rates = pd.crosstab(model_df['storm_active_maui'], model_df['disruption_class'], normalize='index').mul(100).round(2)
    display(storm_rates)


## 10. Save the training-ready v1 table and schema


In [ ]:
save_cols = [c for c in [
    'FlightDate', 'ogg_sched_dt', 'disruption_class', 'split'
] if c in model_df.columns] + feature_cols
save_cols = list(dict.fromkeys(save_cols))

training_ready = model_df[save_cols].copy()
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
training_ready.to_csv(OUT_FILE, index=False, compression='gzip')

schema = {
    'source_file': str(MODEL_FILE),
    'output_file': str(OUT_FILE),
    'rows': int(len(training_ready)),
    'columns': int(training_ready.shape[1]),
    'target': 'disruption_class',
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
    'leakage_excluded': leakage_cols,
    'split_definition': {
        'train': '2020-2023',
        'validation': '2024',
        'test': '2025',
        'holdout_or_uncovered': 'other years, including 2026',
    },
}
SCHEMA_FILE.write_text(json.dumps(schema, indent=2))

print('Saved training table:', OUT_FILE)
print('Shape:', training_ready.shape)
print('Saved schema:', SCHEMA_FILE)


## Decision gate before Notebook 05

Before training the baseline model, record:
- class balance;
- duplicate count;
- weather match rate by year;
- features with high missingness;
- train/validation/test class counts;
- disruption rates during Maui storm intervals.

Notebook 05 will then compare interpretable baselines and tree-based models using the temporal split.
